# Query Decomposition for RAG

An implementation of **Query Decomposition**, a retrieval-augmented generation (RAG) technique
that breaks a complex question into simpler sub-questions before retrieving and answering, then
combines the sub-answers into a final response.

**Why it helps:** a single complex question (e.g. *"What are the main components of an
LLM-powered autonomous agent system?"*) may require pulling together several distinct pieces of
information that no single retrieved chunk covers well. Decomposing it into focused
sub-questions lets each one be answered from the context best suited to it, then the pieces are
recombined into a complete answer.

This notebook implements **two decomposition strategies**:

1. **Recursive answering** — sub-questions are answered in sequence, and each answer is fed as
   context into the next sub-question's prompt (via accumulated Q&A pairs).
2. **Independent answering + synthesis** — sub-questions are answered independently in parallel,
   then all Q&A pairs are combined and a final LLM call synthesizes them into one answer.

**Stack:** LangChain, Chroma (local vector store), Ollama for both an embedding model and a
local chat LLM.

> Source document used in this example: [*LLM Powered Autonomous Agents*](https://lilianweng.github.io/posts/2023-06-23-agent/) by Lilian Weng.


## Setup

Load model names from environment variables (`OLLAMA_MODEL` for the chat model, `OLLAMA_EMBED_LITE` for the embedding model) via a `.env` file, so the notebook doesn't hardcode model choices.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")
EMBED_MODEL = os.getenv("OLLAMA_EMBED_LITE")

## 1. Load Documents

Fetch the source article and strip it down to just the post content, title, and header.

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

C:\Users\chett\AppData\Local\Temp\ipykernel_20320\2459203129.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\chett\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\chett\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


## 2. Chunk the Document

Split the loaded document into overlapping ~500-token chunks so each chunk is small enough to embed and retrieve individually, while the overlap preserves context across chunk boundaries.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_overlap=30, chunk_size=500)
chunks = splitter.split_documents(blog_docs)
print("No. of Chunks: ",len(chunks))
chunks[0]

No. of Chunks:  30


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refin

## 3. Embed & Index

Embed each chunk with a local Ollama embedding model and store the vectors in a Chroma vector store. `retriever` fetches the single most relevant chunk (`k=1`) per query.

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embed = OllamaEmbeddings(model=EMBED_MODEL,num_ctx=1024)
vectorstore = Chroma.from_documents(embedding=embed,documents=chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 4. Decompose the Question

Ask the LLM to break the original question down into a small set of focused sub-questions that can each be answered in isolation.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [13]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatOllama(model=OLLAMA_MODEL,temperature=0,num_ctx=1024)

# Chain
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})
questions

['Here are three search queries related to the main components of an LLM-powered autonomous agent system:',
 '',
 '1. "LLM-based decision-making architectures"',
 '2. "Autonomous agent system design for natural language understanding"',
 '3. "Components of a self-driving autonomous vehicle powered by large language models"',
 '',
 'These queries can help break down the main components of an LLM-powered autonomous agent system into more specific and answerable sub-questions.']

## 5a. Strategy 1 — Recursive Answering

Answer each sub-question in turn, carrying forward the accumulated question/answer pairs as background context for the next sub-question. This lets later sub-questions build on earlier findings, at the cost of being sequential.

In [7]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [14]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

q_a_pairs = ""

for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

q_a_pairs

'\n---\nQuestion: Here are three search queries related to the main components of an LLM-powered autonomous agent system:\nAnswer: Based on the provided context, I\'ll attempt to answer the question by suggesting three search queries related to the main components of an LLM-powered autonomous agent system.\n\n1. "LLM-powered autonomous agent system planning and subgoal decomposition"\n\nThis search query is related to the "Planning" component of the LLM-powered autonomous agent system, which involves breaking down large tasks into smaller, manageable subgoals.\n\n2. "LLM-powered autonomous agent system short-term and long-term memory"\n\nThis search query is related to the "Memory" component of the LLM-powered autonomous agent system, which includes short-term memory for in-context learning and long-term memory for retaining and recalling information over extended periods.\n\n3. "LLM-powered autonomous agent system tool use and external API integration"\n\nThis search query is related 

## 5b. Strategy 2 — Independent Answering + Synthesis

Alternatively, answer every sub-question independently (each retrieves and answers from its own context, with no dependency on the others), then combine all the Q&A pairs and make one final LLM call to synthesize them into a single coherent answer.

In [9]:
# Answer each sub-question independently, then synthesize afterward
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_rag = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n\nContext: {context}\n\nAnswer:"
)

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """Retrieve context and answer each sub-question independently."""

    sub_questions = sub_question_generator_chain.invoke({"question": question})

    rag_results = []
    for sub_question in sub_questions:
        retrieved_docs = retriever.invoke(sub_question)
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)

    return rag_results, sub_questions

answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)


## 6. Synthesize the Final Answer

Format all independently-answered sub-question/answer pairs into one context block, and ask the LLM to synthesize a final answer to the original question from them.

In [10]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

"Based on the provided Q&A pairs, the main components of an LLM-powered autonomous agent system can be synthesized as follows:\n\nThe main components of an LLM-powered autonomous agent system include:\n\n1. Planning: This involves subgoal and decomposition, reflection, and refinement. Planning is a critical component of autonomous decision-making, allowing the agent to break down complex tasks into smaller, manageable subgoals and adjust its approach as needed.\n2. Memory: This includes short-term memory for in-context learning and long-term memory for retaining and recalling information. The agent's memory plays a crucial role in learning from its environment and adapting to new situations.\n3. Tool use: This involves learning to call external APIs for extra information. The agent's ability to utilize external tools and resources enables it to gather additional information, access specialized knowledge, and improve its decision-making capabilities.\n\nThese components work together to

## Next Steps

- Compare the recursive and independent-synthesis strategies quantitatively on a set of multi-part questions to see which handles inter-dependent sub-questions better.
- Combine decomposition with [RAG-Fusion](#) (reciprocal rank fusion across generated queries) for retrieval that's both broader and better-ranked.
- Add evaluation (e.g. answer faithfulness or completeness scoring) to quantify decomposition's benefit over answering the original complex question directly.